# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

I inspected the distributions of search volume, impressions, days since last update, CTR, average position, and word count. Traffic and volume fields are strongly right-skewed, so I use log1p transforms or buckets rather than relying on raw-value correlations.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Load the prepared FlyRank feature vector
# ---------------------------------------------------------

REPO_ROOT = Path.cwd().parents[1]

DATA_PATH = REPO_ROOT / "data/processed/refresh_feature_vector.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)

# Fields used in this audit
audit_fields = [
    "search_volume",
    "impressions_90d",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "word_count",
]

# Basic numeric cleanup
for col in audit_fields:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nDistribution summary:")
display(
    df[audit_fields].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).T
)

# Log summaries for heavy-tailed traffic fields
for col in ["search_volume", "impressions_90d", "word_count"]:
    x = df[col].dropna()
    print(
        f"{col}: median={x.median():.2f}, "
        f"p95={x.quantile(.95):.2f}, "
        f"max={x.max():.2f}"
    )

# Position zero means "no data", so exclude it when describing position.
valid_position = df.loc[df["avg_position"] > 0, "avg_position"]

print(
    f"\nValid avg_position rows: {len(valid_position):,}; "
    f"zero/no-data rows: {(df['avg_position'] == 0).sum():,}"
)

Shape: (30000, 52)

Distribution summary:


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
search_volume,30000.0,145.811667,1455.132022,0.0,0.0,10.00,20.00,110.00,320.00,2400.000,74000.0
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,12136.40,22996.50,73505.830,517715.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,104.00,104.00,106.000,373.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,0.65,1.09,8.330,100.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,36.80,48.20,69.901,245.0
word_count,30000.0,2310.205433,1846.788556,0.0,0.0,2605.00,3247.00,4719.20,5876.15,7090.080,9546.0


search_volume: median=10.00, p95=320.00, max=74000.00
impressions_90d: median=731.00, p95=22996.50, max=517715.00
word_count: median=2605.00, p95=5876.15, max=9546.00

Valid avg_position rows: 28,795; zero/no-data rows: 1,205


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

I tested three observable signals: search volume versus observed impressions, word count versus observed impressions, and CTR versus average position. Each test uses grouped results or a rank-based relationship so that heavy-tailed traffic does not dominate the conclusion.

In [2]:
# =========================================================
# SIGNAL TEST #1
# Claim: Higher search volume means higher observed impressions.
# =========================================================

signal1 = df[
    ["search_volume", "impressions_90d"]
].dropna().copy()

# Four equally populated search-volume buckets
signal1["volume_bucket"] = pd.qcut(
    signal1["search_volume"],
    q=4,
    duplicates="drop"
)

test1 = (
    signal1
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("impressions_90d", "size"),
        median_impressions_90d=("impressions_90d", "median"),
        mean_impressions_90d=("impressions_90d", "mean")
    )
    .reset_index()
)

print("SIGNAL TEST #1")
print("Claim: Higher search volume means higher observed impressions.")
display(test1)

# Spearman is safer than raw Pearson for heavy-tailed traffic.
spearman_1 = signal1[
    ["search_volume", "impressions_90d"]
].corr(method="spearman").iloc[0, 1]

print(f"Spearman correlation: {spearman_1:.4f}")

if spearman_1 >= 0.30:
    verdict_1 = "CONFIRMED"
elif spearman_1 <= -0.30:
    verdict_1 = "OPPOSITE"
elif abs(spearman_1) < 0.10:
    verdict_1 = "FALSE"
else:
    verdict_1 = "MIXED"

print("VERDICT:", verdict_1)

SIGNAL TEST #1
Claim: Higher search volume means higher observed impressions.


,volume_bucket,n,median_impressions_90d,mean_impressions_90d
0,"(-0.001, 10.0]",20860,659.0,4950.883941
1,"(10.0, 20.0]",2290,1006.5,5690.318341
2,"(20.0, 74000.0]",6850,842.5,5796.309635


Spearman correlation: 0.0719
VERDICT: FALSE


In [3]:
# =========================================================
# SIGNAL TEST #2
# Claim: Longer content receives more observed impressions.
# =========================================================

signal2 = df[
    ["word_count", "impressions_90d"]
].dropna().copy()

signal2 = signal2[signal2["word_count"] > 0]

signal2["word_count_bucket"] = pd.qcut(
    signal2["word_count"],
    q=4,
    duplicates="drop"
)

test2 = (
    signal2
    .groupby("word_count_bucket", observed=True)
    .agg(
        n=("impressions_90d", "size"),
        median_impressions_90d=("impressions_90d", "median"),
        mean_impressions_90d=("impressions_90d", "mean")
    )
    .reset_index()
)

print("SIGNAL TEST #2")
print("Claim: Longer content receives more observed impressions.")
display(test2)

spearman_2 = signal2[
    ["word_count", "impressions_90d"]
].corr(method="spearman").iloc[0, 1]

print(f"Spearman correlation: {spearman_2:.4f}")

if spearman_2 >= 0.30:
    verdict_2 = "CONFIRMED"
elif spearman_2 <= -0.30:
    verdict_2 = "OPPOSITE"
elif abs(spearman_2) < 0.10:
    verdict_2 = "FALSE"
else:
    verdict_2 = "MIXED"

print("VERDICT:", verdict_2)

SIGNAL TEST #2
Claim: Longer content receives more observed impressions.


,word_count_bucket,n,median_impressions_90d,mean_impressions_90d
0,"(7.999, 2413.0]",5576,91.0,1370.850430
1,"(2413.0, 2877.0]",5586,1096.5,5396.791801
2,"(2877.0, 3666.0]",5566,889.0,5983.647862
3,"(3666.0, 9546.0]",5573,1495.0,7565.616365


Spearman correlation: 0.2986
VERDICT: MIXED


In [4]:
# =========================================================
# SIGNAL TEST #3
# Claim: Pages with better average position have higher CTR.
# =========================================================

signal3 = df[
    ["avg_position", "ctr", "impressions_90d", "clicks_90d"]
].copy()

# avg_position == 0 means no position data
signal3 = signal3[
    signal3["avg_position"] > 0
].dropna(
    subset=["avg_position", "ctr", "impressions_90d", "clicks_90d"]
)

# Position buckets
signal3["position_bucket"] = pd.cut(
    signal3["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=[
        "top_3",
        "page_1",
        "positions_11_20",
        "deep"
    ]
)

test3 = (
    signal3
    .groupby("position_bucket", observed=True)
    .agg(
        n=("impressions_90d", "size"),
        impressions=("impressions_90d", "sum"),
        clicks=("clicks_90d", "sum")
    )
    .reset_index()
)

# Weighted CTR = total clicks / total impressions
test3["weighted_ctr_pct"] = (
    test3["clicks"] /
    test3["impressions"].replace(0, np.nan)
    * 100
)

print("SIGNAL TEST #3")
print("Claim: Better average position has higher CTR.")
display(test3)

# Spearman check
spearman_3 = signal3[
    ["avg_position", "ctr"]
].corr(method="spearman").iloc[0, 1]

print(f"Spearman correlation between position and CTR: {spearman_3:.4f}")

# Better position = smaller position number.
if spearman_3 <= -0.30:
    verdict_3 = "CONFIRMED"
elif spearman_3 >= 0.30:
    verdict_3 = "OPPOSITE"
elif abs(spearman_3) < 0.10:
    verdict_3 = "FALSE"
else:
    verdict_3 = "MIXED"

print("VERDICT:", verdict_3)

SIGNAL TEST #3
Claim: Better average position has higher CTR.


,position_bucket,n,impressions,clicks,weighted_ctr_pct
0,top_3,1141,7560663,37042,0.489931
1,page_1,11842,89361420,311928,0.349063
2,positions_11_20,7273,22819980,79552,0.348607
3,deep,8539,36266655,54391,0.149975


Spearman correlation between position and CTR: -0.2342
VERDICT: MIXED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked test: staleness and the refresh-candidate flag

FlyRank's refresh logic uses staleness as an important signal. I test whether the existing `is_initial_refresh_candidate` flag is more common among pages with longer `days_since_last_update`. The flag is used only as the audit outcome, not as an input to a model or score.

In [6]:
# =========================================================
# 3. FLAG-LINKED TEST
# Staleness -> freshness tier
# =========================================================

# Check the actual columns before running the audit
print("Available staleness-related columns:")
print([
    c for c in df.columns
    if any(x in c.lower() for x in [
        "fresh", "stale", "age", "update", "refresh"
    ])
])

# ---------------------------------------------------------
# Use the actual freshness_tier field available in the
# prepared feature vector.
# ---------------------------------------------------------

flag_test = df[
    [
        "days_since_last_update",
        "freshness_tier"
    ]
].copy()

flag_test = flag_test.dropna(
    subset=[
        "days_since_last_update",
        "freshness_tier"
    ]
)

# Create staleness buckets
flag_test["staleness_bucket"] = pd.cut(
    flag_test["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, 365, np.inf],
    labels=[
        "0-30",
        "31-90",
        "91-180",
        "181-365",
        "365+"
    ]
)

# Count observations by bucket and freshness tier
flag_table = pd.crosstab(
    flag_test["staleness_bucket"],
    flag_test["freshness_tier"]
)

# Add total n
flag_table["n"] = flag_table.sum(axis=1)

print("FLAG-LINKED TEST")
print(
    "Claim: Increasing staleness should be reflected "
    "in the freshness classification."
)

display(flag_table)

# ---------------------------------------------------------
# Show the dominant freshness tier in each bucket
# ---------------------------------------------------------

freshness_summary = (
    flag_test
    .groupby("staleness_bucket", observed=True)
    .agg(
        n=("freshness_tier", "size"),
        median_days_since_update=(
            "days_since_last_update",
            "median"
        ),
        dominant_freshness_tier=(
            "freshness_tier",
            lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
        )
    )
    .reset_index()
)

print("\nFreshness summary:")
display(freshness_summary)

# ---------------------------------------------------------
# Check sample sizes
# ---------------------------------------------------------

print("\nBuckets with n < 50:")
display(
    freshness_summary[
        freshness_summary["n"] < 50
    ]
)

# ---------------------------------------------------------
# Simple directional check:
# compare median staleness of each freshness tier
# ---------------------------------------------------------

tier_summary = (
    flag_test
    .groupby("freshness_tier", observed=True)
    .agg(
        n=("days_since_last_update", "size"),
        median_days_since_update=(
            "days_since_last_update",
            "median"
        ),
        mean_days_since_update=(
            "days_since_last_update",
            "mean"
        )
    )
    .reset_index()
    .sort_values("median_days_since_update")
)

print("\nFreshness tiers ordered by staleness:")
display(tier_summary)

# ---------------------------------------------------------
# Verdict
# ---------------------------------------------------------

if len(tier_summary) >= 2:
    medians = tier_summary["median_days_since_update"].values

    if np.all(np.diff(medians) >= 0):
        flag_verdict = "CONFIRMED"
    elif np.all(np.diff(medians) <= 0):
        flag_verdict = "OPPOSITE"
    else:
        flag_verdict = "MIXED"
else:
    flag_verdict = "MIXED"

print("VERDICT:", flag_verdict)

Available staleness-related columns:
['pageviews_90d', 'engaged_sessions_90d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'engagement_rate']
FLAG-LINKED TEST
Claim: Increasing staleness should be reflected in the freshness classification.


freshness_tier,0-30,181+,31-90,91-180,n
staleness_bucket,,,,,
0-30,20480,0,0,0,20480
31-90,0,0,175,0,175
91-180,0,0,0,9171,9171
181-365,0,169,0,0,169
365+,0,5,0,0,5



Freshness summary:


,staleness_bucket,n,median_days_since_update,dominant_freshness_tier
0,0-30,20480,20.0,0-30
1,31-90,175,41.0,31-90
2,91-180,9171,104.0,91-180
3,181-365,169,211.0,181+
4,365+,5,373.0,181+



Buckets with n < 50:


,staleness_bucket,n,median_days_since_update,dominant_freshness_tier
4,365+,5,373.0,181+



Freshness tiers ordered by staleness:


,freshness_tier,n,median_days_since_update,mean_days_since_update
0,0-30,20480,20.0,18.540918
2,31-90,175,41.0,53.628571
3,91-180,9171,104.0,104.106204
1,181+,174,211.0,224.643678


VERDICT: CONFIRMED


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The audit shows which intuitive signals are actually supported by the observed data. Search volume should not be treated as a direct proxy for impressions when the measured relationship is weak, while CTR and position show a much clearer directional relationship. The flag-linked staleness test gives a direct check on whether the refresh-candidate logic is consistent with the observed age signal. These results are decision-support evidence, not causal claims about Google's ranking system.

In [7]:
# Compact audit summary

summary = pd.DataFrame({
    "signal": [
        "search_volume -> impressions_90d",
        "word_count -> impressions_90d",
        "avg_position -> CTR",
        "days_since_last_update -> initial_refresh_candidate"
    ],
    "verdict": [
        verdict_1,
        verdict_2,
        verdict_3,
        flag_verdict
    ]
})

display(summary)

,signal,verdict
0,search_volume -> impressions_90d,FALSE
1,word_count -> impressions_90d,MIXED
2,avg_position -> CTR,MIXED
3,days_since_last_update -> initial_refresh_cand...,CONFIRMED


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.